# Phase 16 — Ensemble BERT-base (Focal Loss) + DistilBERT

**Μοντέλα:**
- BERT-base + Focal Loss, train+valid 20 epochs → Kaggle: 0.80399
- DistilBERT, train+valid 20 epochs → Kaggle: 0.76064

**Μέθοδος:** Weighted average των probabilities

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# BERT-base + Focal Loss probs (best: 0.80399)
bertbase_hazard_probs  = np.load('npy/bertbase_focal_test_hazard_probs2.npy')
bertbase_product_probs = np.load('npy/bertbase_focal_test_product_probs2.npy')

# DistilBERT probs (0.76064)
distil_hazard_probs  = np.load('npy/bert_fulldata_test_hazard_probs.npy')
distil_product_probs = np.load('npy/bert_fulldata_test_product_probs.npy')

print(f'BERT-base hazard probs:  {bertbase_hazard_probs.shape}')
print(f'DistilBERT hazard probs: {distil_hazard_probs.shape}')

BERT-base hazard probs:  (997, 10)
DistilBERT hazard probs: (997, 10)


In [3]:
# Weighted average — δίνουμε περισσότερο βάρος στο BERT-base
# γιατί έχει καλύτερο Kaggle score (0.804 vs 0.761)
weight_combinations = [
    (1.0, 0.0),   # BERT-base μόνο (baseline)
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.0, 1.0),   # DistilBERT μόνο (baseline)
]

print('Δημιουργία submissions\n')

for w_bert, w_distil in weight_combinations:
    hazard_avg  = w_bert * bertbase_hazard_probs  + w_distil * distil_hazard_probs
    product_avg = w_bert * bertbase_product_probs + w_distil * distil_product_probs

    test_hazard  = le_hazard.inverse_transform(hazard_avg.argmax(axis=1))
    test_product = le_product.inverse_transform(product_avg.argmax(axis=1))

    filename = f'submission_bert{int(w_bert*10)}_distil{int(w_distil*10)}.csv'
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_hazard,
        'product-category': test_product
    }).to_csv(filename, index=False)
    print(f'BERT-base={w_bert:.1f}, DistilBERT={w_distil:.1f} → {filename}')

print('\nΌλα τα submissions έτοιμα')

Δημιουργία submissions

BERT-base=1.0, DistilBERT=0.0 → submission_bert10_distil0.csv
BERT-base=0.9, DistilBERT=0.1 → submission_bert9_distil1.csv
BERT-base=0.8, DistilBERT=0.2 → submission_bert8_distil2.csv
BERT-base=0.7, DistilBERT=0.3 → submission_bert7_distil3.csv
BERT-base=0.6, DistilBERT=0.4 → submission_bert6_distil4.csv
BERT-base=0.5, DistilBERT=0.5 → submission_bert5_distil5.csv
BERT-base=0.0, DistilBERT=1.0 → submission_bert0_distil10.csv

Όλα τα submissions έτοιμα
